# 1. Tensors and data flow
### 1.1 Tensor

#### What it is  
A tensor is the core data structure in deep learning: a multi-dimensional array with a defined shape and data type. In PyTorch, everything (inputs, weights, activations, gradients) is a tensor.

For your segmentation model, the main tensor shapes are:

- Input images: (B,C,H,W)

    - B: batch size (number of patches in one step)

    - C: channels (e.g. 8: DEM + derivatives)

    - H,W: spatial dimensions of the patch

- Masks: (B,H,W)

    - Each pixel holds an integer class index [0,num_classes−1]

- Model outputs (logits): (B,num_classes,H,W)

#### Why it matters  
Understanding tensor shapes is the foundation. Every layer you use (convolutions, pooling, upsampling) transforms these shapes in specific ways. If you know the shapes, you can reason about the architecture.

 ---
### 1.2 Batch

What it is  
A batch is a group of samples processed together in one forward/backward pass.

Why it matters

- It stabilizes gradient estimates (compared to single samples).

- It allows efficient use of GPU parallelism.

- Many operations (BatchNorm, loss) are defined over the batch dimension.

 ---
 ---
 ---
# 2. Convolution and feature extraction

Now we follow the first thing that happens to your input tensor inside the model: convolutions.

### 2.1 Convolution (Conv2D)

#### What it is  
A 2D convolution applies learnable filters (kernels) over the spatial dimensions of the input. Each filter “looks” at a local neighborhood and produces one output channel.

Formally, for one output channel:
y(i,j)=∑c=1Cin∑u,vWc,u,v⋅xc(i+u,j+v)+b

Where:

- x is the input tensor

- W is the kernel (weights)

- b is the bias

- Cin is the number of input channels

Intuition

- Early layers learn edge/texture detectors.

- Deeper layers learn more abstract patterns (shapes, regions, structures).

Variants

- Standard Conv2D 

- Dilated convolutions (larger receptive field without pooling)

- Depthwise separable convolutions (more efficient)

U-Net ConvBlock is essentially:

- Conv2D → BatchNorm → Activation → Conv2D → BatchNorm → Activation

This is the basic feature extraction unit used in both encoder and decoder.

 ---
### 2.2 Padding

#### What it is  
Padding adds extra pixels around the border of the input (often zeros) so that the spatial size is preserved after convolution.

Without padding, a 3×3 convolution reduces H,W by 2.

Why it matters in UNet

- You want encoder and decoder feature maps to align spatially for skip connections.

- Using padding=1 with kernel_size=3 keeps H,W unchanged.

 ---
### 2.3 Stride

#### What it is  
Stride is how far the convolution kernel moves each step.

- Stride = 1 → dense scanning

- Stride > 1 → downsampling

You mostly use stride = 1 in convolutions, and use pooling for downsampling instead.

 ---
 ---
 ---
# 3. Activation functions

After each convolution, you apply an activation function. This is where non-linearity enters the model.

### 3.1 Why activation functions exist

Without activations, a stack of linear layers (convolutions) is still just a linear function. It cannot model complex, non-linear relationships.

Activation functions allow the network to approximate arbitrary non-linear mappings.

 ---
### 3.2 ReLU

#### Definition
ReLU(x)=max⁡(0,x)

Properties

- Simple and fast

- Encourages sparsity (many zeros)

- But can cause “dead neurons” if weights push activations permanently negative.

 ---
### 3.3 LeakyReLU

#### Definition
LeakyReLU(x)={xx>0αxx≤0}

with α typically small (e.g. 0.01 or 0.1).

Why it might be preferable

- It keeps a small gradient for negative values.

- Reduces the risk of dead neurons.

- Can be more robust when inputs are noisy or have wide value ranges.

 ---
 ---
 ---
# 4. Normalization and regularization inside the model

Now we look at the “stabilizers” inside the blocks: BatchNorm and Dropout.

### 4.1 Batch Normalization (BatchNorm2d)

#### What it does  
For each channel, over the batch and spatial dimensions, it normalizes activations to have mean 0 and variance 1, then scales and shifts them with learnable parameters.

Formally, for each channel:
$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \qquad
y = \gamma \hat{x} + \beta
$$


Why it matters

- Reduces internal covariate shift (distribution drift of activations).

- Allows higher learning rates.

- Often speeds up and stabilizes training.

ConvBlock uses BatchNorm after convolutions. This helps when your input channels have very different distributions.

 ---
### 4.2 Dropout2D

#### What it does  
Randomly zeroes entire feature channels (not just individual pixels) during training.

Why it matters

- Forces the network not to rely too heavily on any single channel.

- Reduces overfitting, especially when positive class is extremely rare.

- In 2D feature maps, dropping whole channels is more structured than dropping random pixels.
  
You can use Dropout2d in ConvBlock. This is particularly relevant when dataset is highly imbalanced: most pixels are background for example.

 ---
 ---
 ---
# 5. Encoder: downsampling and hierarchical features

Now we follow the data as it moves deeper into the network.

### 5.1 Max Pooling (MaxPool2d)

#### What it does  
For each channel, it takes the maximum value in non-overlapping windows (e.g. 2×2), reducing spatial resolution by a factor of 2.

#### Why it matters

- Increases the receptive field (each deeper neuron “sees” a larger part of the input).

- Keeps the most salient activations.

- Reduces memory and computation.

 ---
### 5.2 EncoderBlock

The EncoderBlock does:

- ConvBlock(in_channels, out_channels)

- MaxPool2d(kernel_size=2, stride=2)

It returns:

- skip: output of ConvBlock (high-resolution features)

- pooled: output after pooling (lower resolution, deeper features)

#### Why this structure?

- skip is used later in the decoder to recover fine spatial details.

- pooled is passed deeper into the network to build more abstract representations.

#### Big picture  
The encoder gradually transforms the input into a hierarchy of features:

- Shallow layers: local edges, textures, small patterns

- Deeper layers: shapes, regions, global context

 ---
 ---
 ---
# 6. Bottleneck: deepest representation
### 6.1 Bottleneck ConvBlock

At the bottom of the UNet, you have:

- self.bottleneck = ConvBlock(512, 1024)

What it represents

- Lowest spatial resolution

- Highest number of channels

- Most abstract representation of the input

#### Why it matters

- This is where the model can integrate global context.

- For segmentation, this helps understand “where” structures are in relation to the whole patch.

 ---
 ---
 ---
# 7. Decoder: upsampling and reconstruction

Now we go back up: the decoder reconstructs a high-resolution segmentation map.

### 7.1 Upsampling

You have two main options conceptually:

- Bilinear upsampling (nn.Upsample)

    - Non-learnable

    - Smooth interpolation

    - No checkerboard artifacts

- Transposed convolution (nn.ConvTranspose2d)

    - Learnable

    - Can produce sharper boundaries

    - Risk of checkerboard artifacts if not carefully designed

You can create a DecoderBlock supporting both (via bilinear=True/False).

 ---
### 7.2 Spatial alignment and padding

After upsampling, the spatial size of the decoder feature map might not perfectly match the skip connection (due to odd dimensions, pooling, etc.).

You handle this by computing:

- diff_y = skip.size(2) - x.size(2)

- diff_x = skip.size(3) - x.size(3)

And then padding x so it matches skip.

#### Why this matters

- Concatenation requires exact spatial alignment.

- Without this, you’d get shape mismatch errors.

- In geospatial data, odd tile sizes are common, so this is practically important.

 ---
### 7.3 Skip connections and concatenation

Once sizes match, you do:

- x = torch.cat([x, skip], dim=1)

What this does

- Concatenates the upsampled decoder features with the encoder’s high-resolution features along the channel dimension.

#### Why it’s powerful

- Decoder alone would only have coarse, abstract features.

- Skip connections inject fine-grained spatial detail from the encoder.

- This combination allows UNet to produce sharp segmentations even after heavy downsampling.

 ---
### 7.4 DecoderBlock

So a DecoderBlock is:

- Upsample decoder features

- Align with skip (padding)

- Concatenate with skip

- Apply ConvBlock to fuse and refine

This pattern is repeated until you reach the original resolution.

 ---
 ---
 ---
# 8. Output layer and logits
### 8.1 1×1 Convolution

The final layer is:

self.output = nn.Conv2d(64, num_classes, kernel_size=1)

#### What it does

- For each pixel, it takes the 64-dimensional feature vector and linearly maps it to num_classes logits.

#### Why 1×1?

- It doesn’t mix spatial information, only channels.

- It acts as a per-pixel classifier on top of the learned features.

 ---
### 8.2 Logits and softmax

The model returns raw logits: (B,num_classes,H,W).

- During training, these logits are passed directly to the loss function (e.g. CrossEntropy, Focal).

- The loss function internally applies log_softmax or similar.

- During inference, you typically apply softmax and then argmax to get class predictions.

 ---
 ---
 ---
# 9. Loss functions and class imbalance

Now we move outside the model into the training loop.

### 9.1 CrossEntropyLoss

What it assumes

- Classes are reasonably balanced.

- Every pixel is equally important.

It fails for thin objects becouse

- Background dominates.

- The model can get high accuracy by predicting background everywhere.

- It doesn’t care about the shape or continuity of small structures.

 ---
### 9.2 Dice loss

#### What it measures

Dice coefficient measures overlap between prediction and ground truth:

- Dice=2∣P∩G∣∣P∣+∣G∣

Dice loss is 1−Dice.

Why it’s good for thin objects

- It directly optimizes overlap, not per-pixel accuracy.

- It is less sensitive to class imbalance.

 ---
### 9.3 Focal loss

#### What it does

Focal loss modifies CrossEntropy to focus more on hard examples:

$$
\text{FL} = (1 - p_t)^{\gamma} \cdot \text{CE}
$$


Where pt is the predicted probability of the true class.

#### Why it helps

- Easy examples (background) are down-weighted.

- Hard examples (rivers) contribute more to the loss.

 ---
### 9.4 Combined Dice + Focal

In the loop, you can use:

- loss = 0.5 * focal(pred, target) + 0.5 * dice_loss(pred, target)

#### Why this is powerful

- Dice focuses on overlap and shape.

- Focal focuses on hard pixels and class imbalance.

- Together, they encourage the model to both detect and correctly shape thin structures.

 ---
 ---
 ---
# 10. Training loop mechanics
### 10.1 Forward pass

For each batch:

- Load images and masks

- Move them to device

- Compute outputs = model(images)

- Compute loss = loss_fn(outputs, masks)

This is the forward pass: data flows through the model, loss is computed.

 ---
### 10.2 Backward pass and optimization

Then:

- optimizer.zero_grad() — clear old gradients

- loss.backward() — compute gradients via backpropagation

- optimizer.step() — update parameters using gradients

Backpropagation

- Uses the chain rule to propagate gradients from the loss back through all layers.

- Each parameter gets a gradient telling it how to change to reduce the loss.

 ---
### 10.3 Epochs and validation

An epoch is one full pass over the training dataset.

At the end of each epoch, you:

- Run a validation loop (no gradients)

- Compute validation loss and Dice score

- Optionally save a checkpoint if validation loss improves

This gives you a signal of generalization, not just training performance.

 ---
 ---
 ---
# 11. Dataset and DataLoader
### 11.1 Dataset

The PatchDataset:

- Scans folders for img*.npy and mask*.npy pairs

- Loads them as NumPy arrays

- Converts them to PyTorch tensors

- Returns (image, mask) for each index

This defines what one sample is.

 ---
### 11.2 DataLoader

DataLoader wraps the dataset and handles:

- Batching

- Shuffling

- Parallel loading (num_workers)

- Pinned memory for faster GPU transfer

This defines how samples are fed into the model.

 ---
 ---
 ---
# 12. The big picture: how it all connects

Let’s tie it all together in one flow:

- Dataset loads raw patches → tensors (B,C,H,W) and masks (B,H,W).

- DataLoader batches them and feeds them to the model.

- UNet encoder:

    - ConvBlocks extract local features.

    - BatchNorm stabilizes activations.

    - LeakyReLU introduces non-linearity.

    - Dropout2D regularizes.

    - MaxPool downsamples and builds a feature hierarchy.

- Bottleneck integrates global context at low resolution.

- UNet decoder:

    - Upsampling restores spatial resolution.

    - Skip connections inject high-resolution details.

    - ConvBlocks fuse deep and shallow features.

- Output layer (1×1 conv) maps features to class logits per pixel.

- Loss functions (Dice + Focal) compare logits to ground truth masks, focusing on overlap and hard pixels.

- Backpropagation computes gradients through all layers.

- Optimizer (Adam) updates weights to reduce loss.

- Validation measures performance (loss + Dice) on unseen patches.

- Checkpoints save the best model based on validation loss.

 ---
 ---
 ---
 # 13. Heavily commented example of a dataloader for U-Net

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset


# ============================================================
# (1) PatchDataset – structured loading of image/mask pairs
# ------------------------------------------------------------
# This dataset class:
#   - scans a root directory for patch folders
#   - pairs each image patch with its corresponding mask patch
#   - loads them as NumPy arrays
#   - converts them into PyTorch tensors
#   - ensures correct shapes and dtypes for segmentation
#
# The design mirrors the conceptual flow:
#   filesystem → numpy arrays → tensors → model input
# ============================================================
class PatchDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        (1.1) Initialization:
              - root_dir: directory containing subfolders of patches
              - transform: optional augmentation function
        """
        self.root_dir = root_dir
        self.transform = transform

        # (1.2) Internal list storing (image_path, mask_path) pairs
        self.samples = []

        # (1.3) Scan all subdirectories for patch files
        for folder in os.listdir(root_dir):
            folder_path = os.path.join(root_dir, folder)

            # Skip non-directories
            if not os.path.isdir(folder_path):
                continue

            # Find all image patches in this folder
            img_files = sorted([
                f for f in os.listdir(folder_path)
                if f.startswith("img") and f.endswith(".npy")
            ])

            # For each image patch, find its corresponding mask
            for img_file in img_files:
                # Extract index from filename (e.g. img_12.npy → "12")
                base = img_file.replace("img", "").replace(".npy", "")
                index = base[1:] if base.startswith("_") else ""

                mask_file = f"mask{('_' + index) if index else ''}.npy"
                mask_path = os.path.join(folder_path, mask_file)

                # Only add sample if mask exists
                if os.path.exists(mask_path):
                    self.samples.append((
                        os.path.join(folder_path, img_file),
                        mask_path
                    ))

    # ============================================================
    # (2) __len__ – number of samples
    # ------------------------------------------------------------
    # Required by PyTorch. Allows DataLoader to know dataset size.
    # ============================================================
    def __len__(self):
        return len(self.samples)

    # ============================================================
    # (3) __getitem__ – load and prepare a single sample
    # ------------------------------------------------------------
    # This method performs:
    #   - file loading
    #   - shape correction
    #   - dtype conversion
    #   - optional augmentation
    #
    # Output format:
    #   image: (C, H, W) float32 tensor
    #   mask:  (H, W)   long tensor (class indices)
    #
    # This matches exactly what the UNet and training loop expect.
    # ============================================================
    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        # (3.1) Load raw NumPy arrays
        image = np.load(img_path).astype(np.float32)   # shape: (C, H, W)
        mask  = np.load(mask_path).astype(np.int64)    # shape: (H, W) or (1, H, W)

        # (3.2) Ensure mask is 2D (remove channel dimension if present)
        if mask.ndim == 3:
            mask = mask.squeeze(0)

        # (3.3) Safety clamp:
        #       Ensures mask values are within valid class range.
        #       Prevents crashes in loss functions.
        mask = np.clip(mask, 0, mask.max())

        # (3.4) Convert to PyTorch tensors
        image = torch.tensor(image, dtype=torch.float32)
        mask  = torch.tensor(mask, dtype=torch.long)

        # (3.5) Optional augmentations (e.g. flips, rotations)
        if self.transform is not None:
            image, mask = self.transform(image, mask)

        # (3.6) Return final tensors
        return image, mask


---
 ---
 ---
 # 14. Heavily commented example of a U-Net architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# (1) ConvBlock – core feature extraction unit
# ------------------------------------------------------------
# This block is the fundamental building stone of the network.
# It:
#   - applies two convolutions
#   - normalizes activations
#   - applies non-linear activation
#
# Conceptually:
#   Input tensor:  (B, C_in, H, W)
#   Output tensor: (B, C_out, H, W)
#
# This is where the model learns local patterns (edges, textures,
# shapes) that later get combined into more abstract features.
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.LeakyReLU(0.1, inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        # First conv → normalize → non-linearity
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act1(x)

        # Second conv → normalize → non-linearity
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act2(x)

        return x


# ============================================================
# (2) EncoderStage – downsampling with feature extraction
# ------------------------------------------------------------
# Each encoder stage:
#   - refines features at current resolution (ConvBlock)
#   - reduces spatial size (MaxPool)
#
# It returns:
#   - features_before_pool: used later as skip connection
#   - features_after_pool: passed deeper into the network
#
# This is the "contracting path" of UNet.
# ============================================================
class EncoderStage(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = ConvBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # Extract features at current resolution
        features = self.block(x)

        # Downsample to get more abstract, lower-resolution representation
        down = self.pool(features)

        # Return both:
        #   - features: for skip connection
        #   - down: for deeper encoder / bottleneck
        return features, down


# ============================================================
# (3) DecoderStage – upsampling and skip fusion
# ------------------------------------------------------------
# Each decoder stage:
#   - upsamples the input feature map
#   - aligns it with the corresponding encoder feature map
#   - concatenates them along the channel dimension
#   - refines the fused features with a ConvBlock
#
# Here we use ConvTranspose2d for learnable upsampling instead
# of simple bilinear upsampling. This gives the decoder its 
# parameters for how to reconstruct spatial detail.
# ============================================================
class DecoderStage(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        # Transposed convolution doubles spatial resolution
        # and reduces channel count from in_channels to out_channels.
        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        # After upsampling, we will concatenate with skip features:
        #   channels = out_channels (from up) + skip_channels
        self.fuse = ConvBlock(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        # 1) Upsample decoder features
        x = self.up(x)

        # 2) If shapes differ slightly (due to pooling/odd sizes),
        #    we crop the skip tensor to match the upsampled tensor.
        #    Padding would also work?
        if x.size(2) != skip.size(2) or x.size(3) != skip.size(3):
            # Compute how much to crop from skip
            diff_y = skip.size(2) - x.size(2)
            diff_x = skip.size(3) - x.size(3)

            # Center-crop skip to match x
            skip = skip[
                :,
                :,
                diff_y // 2 : skip.size(2) - diff_y + diff_y // 2,
                diff_x // 2 : skip.size(3) - diff_x + diff_x // 2,
            ]

        # 3) Concatenate along channel dimension:
        #    - x:   upsampled decoder features
        #    - skip: encoder features with fine spatial detail
        x = torch.cat([skip, x], dim=1)

        # 4) Refine fused features
        x = self.fuse(x)
        return x


# ============================================================
# (4) UNet – full architecture
# ------------------------------------------------------------
# This class ties everything together:
#
#   - Encoder: sequence of EncoderStage modules
#   - Bottleneck: deepest ConvBlock
#   - Decoder: sequence of DecoderStage modules
#   - Output: 1×1 convolution mapping features → class logits
#
# Data flow (high level):
#   input
#     → encoder stages (downsampling, feature hierarchy)
#     → bottleneck (global context)
#     → decoder stages (upsampling, skip fusion)
#     → output layer (per-pixel classification)
#
# The forward method is written to clearly reflect this flow.
# ============================================================
class UNet(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()

        # ---------------- Encoder path ----------------
        # Each stage doubles channels and halves spatial size.
        self.enc1 = EncoderStage(in_channels, 64)
        self.enc2 = EncoderStage(64, 128)
        self.enc3 = EncoderStage(128, 256)
        self.enc4 = EncoderStage(256, 512)

        # ---------------- Bottleneck ----------------
        # Deepest representation with the highest channel count.
        self.bottleneck = ConvBlock(512, 1024)

        # ---------------- Decoder path ----------------
        # Note: in_channels here is the number of channels coming
        # from the previous decoder output (or bottleneck),
        # skip_channels is the number of channels from the encoder.
        self.dec1 = DecoderStage(in_channels=1024, skip_channels=512, out_channels=512)
        self.dec2 = DecoderStage(in_channels=512,  skip_channels=256, out_channels=256)
        self.dec3 = DecoderStage(in_channels=256,  skip_channels=128, out_channels=128)
        self.dec4 = DecoderStage(in_channels=128,  skip_channels=64,  out_channels=64)

        # ---------------- Output layer ----------------
        # 1×1 convolution:
        #   - keeps spatial size
        #   - maps from feature channels → num_classes
        self.classifier = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # ========== ENCODER ==========
        # At each stage:
        #   - features_i: high-res features (for skip)
        #   - down_i:     low-res features (for deeper layers)

        # Stage 1: closest to raw input
        features1, down1 = self.enc1(x)

        # Stage 2: more abstract, lower resolution
        features2, down2 = self.enc2(down1)

        # Stage 3: even deeper representation
        features3, down3 = self.enc3(down2)

        # Stage 4: deepest encoder stage before bottleneck
        features4, down4 = self.enc4(down3)

        # ========== BOTTLENECK ==========
        # Here the model has the smallest spatial size and the
        # richest channel representation. This is where global
        # context is integrated.
        bottleneck = self.bottleneck(down4)

        # ========== DECODER ==========
        # Now we progressively reconstruct spatial detail while
        # reusing encoder features via skip connections.

        # Decoder stage 1:
        #   input: bottleneck (deep features)
        #   skip:  features4 (encoder stage 4)
        x = self.dec1(bottleneck, features4)

        # Decoder stage 2:
        #   input: output of previous decoder
        #   skip:  features3
        x = self.dec2(x, features3)

        # Decoder stage 3:
        #   input: output of previous decoder
        #   skip:  features2
        x = self.dec3(x, features2)

        # Decoder stage 4:
        #   input: output of previous decoder
        #   skip:  features1 (closest to input, finest details)
        x = self.dec4(x, features1)

        # ========== OUTPUT ==========
        # Final per-pixel logits:
        #   shape: (B, num_classes, H, W)
        logits = self.classifier(x)

        return logits


 ---
 ---
 ---
 # 15. Heavily commented example of a U-Net trining loop

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F


# ============================================================
# (1) Dice Loss – measures overlap, not per-pixel accuracy
# ------------------------------------------------------------
# Dice loss is essential for thin objects because:
#   - it ignores true negatives (background)
#   - it directly optimizes spatial overlap
#   - it is robust when positive pixels are extremely rare
#
# This implementation works for multi-class segmentation by
# selecting the predicted probability of the correct class.
# ============================================================
def dice_loss(logits, target, eps=1e-6):
    # Convert logits → probabilities
    probs = F.softmax(logits, dim=1)

    # Gather probability of the correct class for each pixel
    # shape: (B, H, W)
    true_class_probs = probs.gather(1, target.unsqueeze(1)).squeeze(1)

    # Convert target to one-hot for union computation
    one_hot = torch.zeros_like(probs).scatter_(1, target.unsqueeze(1), 1)
    one_hot = one_hot[:, 1:, :, :]  # ignore background if desired

    # Intersection and union
    intersection = (true_class_probs * one_hot[:, 0]).sum()
    union = true_class_probs.sum() + one_hot[:, 0].sum()

    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice


# ============================================================
# (2) Focal Loss – focuses on hard pixels
# ------------------------------------------------------------
# Focal loss reduces the influence of easy background pixels.
# This is crucial when thin objects represent <1% of pixels.
#
# FL = (1 - p_t)^gamma * CE
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, target):
        ce = self.ce(logits, target)
        pt = torch.exp(-ce)  # probability of the correct class
        focal = (1 - pt) ** self.gamma * ce
        return focal


# ============================================================
# (3) Combined Loss – balances shape + difficulty
# ------------------------------------------------------------
# Dice handles shape and overlap.
# Focal handles class imbalance and hard pixels.
#
# The combination is extremely effective for thin structures.
# ============================================================
class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.5, focal_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.focal = FocalLoss()

    def forward(self, logits, target):
        d = dice_loss(logits, target)
        f = self.focal(logits, target)
        return self.dice_weight * d + self.focal_weight * f


# ============================================================
# (4) Validation Step – evaluates generalization
# ------------------------------------------------------------
# Validation uses:
#   - no gradient computation
#   - Dice score (1 - dice_loss)
#   - average loss across batches
#
# Dice score is far more meaningful than pixel accuracy for
# thin objects, because pixel accuracy is dominated by background.
# ============================================================
def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = loss_fn(logits, masks)
            total_loss += loss.item()

            dice_score = 1 - dice_loss(logits, masks)
            total_dice += dice_score.item()

    return total_loss / len(loader), total_dice / len(loader)


# ============================================================
# (5) Training Loop – full pipeline
# ------------------------------------------------------------
# This loop:
#   - loads batches
#   - performs forward pass
#   - computes combined loss
#   - backpropagates gradients
#   - updates weights
#   - evaluates on validation set
#   - saves best model
#
# The comments follow the conceptual flow:
#   data → model → logits → loss → gradients → update
# ============================================================
def train_model(model, train_dataset, val_dataset, num_classes, batch_size=4, epochs=10):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=4)

    # Loss and optimizer
    loss_fn = CombinedLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        print(f"\n=== Epoch {epoch}/{epochs} ===")

        # ---------------- TRAINING ----------------
        model.train()
        running_loss = 0.0

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            # (5.1) Forward pass: model produces logits
            logits = model(images)

            # (5.2) Compute combined loss
            loss = loss_fn(logits, masks)

            # (5.3) Backpropagation
            optimizer.zero_grad()
            loss.backward()

            # (5.4) Weight update
            optimizer.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)

        # ---------------- VALIDATION ----------------
        val_loss, val_dice = validate(model, val_loader, loss_fn, device)

        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print(f"Val Dice:   {val_dice:.4f}")

        # ---------------- CHECKPOINT ----------------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"best_unet_epoch_{epoch}.pth")
            print("Saved new best model.")

    print("\nTraining complete.")
